In [4]:
import argparse
import warnings
from pathlib import Path

import matplotlib
matplotlib.use("Agg")                                   # headless-safe; figures are saved to disk
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (adjusted_rand_score, calinski_harabasz_score,
                             davies_bouldin_score, silhouette_score)
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)


In [5]:
# --------------------------------------------------------------------------- #
# 0. CONFIGURATION
# --------------------------------------------------------------------------- #
SEED = 42
SESSION_GAP_MIN = 30            # inactivity gap (minutes) that ends a session
ACTIVE_WEEK_SHARE = 0.20        # a week is "in course" if >= 20% of learners were active in it
MIN_INTERACTIONS = 20           # learners below this are described but not clustered
K_RANGE = range(2, 9)           # k values scored for the diagnostics plot
K_CANDIDATES = range(3, 7)      # k values we are willing to choose from (>=3 profiles, <=6 readable)
N_BOOT, N_PERM = 100, 200       # bootstrap / permutation repetitions
CSV_NAMES = ["MOODLE Data.csv", "MOODLE_Data.csv"]

# Events generated by the browser / Moodle itself rather than by a deliberate learner action.
SYSTEM_EVENTS = {"Quiz attempt auto-saved", "Submitted SCORM raw score", "Submitted SCORM status",
                 "Read tracking enabled", "Read tracking disabled"}
# Content-code -> behavioural group (anything else with a content code = learning content)
CONTENT_GROUP = {"Continuous Evaluation": "graded_quiz", "Practice Quiz": "practice_quiz",
                 "Assignment Submissions": "assignment", "Course Project": "assignment",
                 "Discussion Forum": "social", "Glossary": "social"}
GROUPS = ["learning_content", "practice_quiz", "graded_quiz", "assignment", "social", "navigation"]

# Features that go into the clustering (chosen to describe HOW, WHEN and HOW REGULARLY, not just how much)
CLUSTER_FEATURES = ["n_interactions", "mean_session_min", "active_week_ratio", "weekly_cv",
                    "longest_inactive_streak", "rush_share", "share_practice_quiz",
                    "share_graded_quiz", "share_learning_content", "night_share",
                    "weekend_share", "first_access_day"]
LOG_FEATURES = ["n_interactions", "mean_session_min", "longest_inactive_streak", "first_access_day"]
FEATURE_FAMILY = {"n_interactions": "volume", "mean_session_min": "volume",
                  "active_week_ratio": "consistency", "weekly_cv": "consistency",
                  "longest_inactive_streak": "consistency", "rush_share": "timing",
                  "first_access_day": "timing", "night_share": "timing", "weekend_share": "timing",
                  "share_practice_quiz": "content mix", "share_graded_quiz": "content mix",
                  "share_learning_content": "content mix"}

sns.set_theme(style="whitegrid", context="notebook")


def find_csv(cli_path):
    if cli_path:
        return Path(cli_path)

    # Safely get current script directory if __file__ is defined
    script_dir = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()

    for base in (Path.cwd(), script_dir, Path("/mnt/user-data/uploads")):
        for name in CSV_NAMES:
            if (base / name).exists():
                return base / name
    raise FileNotFoundError(f"Could not find any of {CSV_NAMES}. Pass --csv path/to/file.csv")

In [6]:
# --------------------------------------------------------------------------- #
# 1. LOADING & CLEANING
# --------------------------------------------------------------------------- #
def load_and_clean(path):
    """Parse, de-duplicate and annotate the raw event log. Returns (df, cleaning_log)."""
    raw = pd.read_csv(path)
    raw.columns = [c.strip() for c in raw.columns]
    df = raw.rename(columns={"User ID": "user", "Time": "time", "Event context": "context",
                             "Component": "component", "Content code": "content",
                             "Event name": "event", "Description": "description"})
    log = {"raw_rows": len(df), "raw_learners": df["user"].nunique()}

    # IDs are exported as quoted strings ('3331') -> strip quotes/whitespace
    df["user"] = df["user"].astype(str).str.strip().str.strip("'\"")
    for c in ["context", "component", "content", "event"]:
        df[c] = df[c].astype("string").str.strip()

    # Timestamps are day/month/2-digit-year, e.g. "29/07/26, 00:42:04"; unparseable rows are dropped
    df["ts"] = pd.to_datetime(df["time"].astype(str).str.strip(), format="%d/%m/%y, %H:%M:%S",
                              errors="coerce")
    log["unparseable_timestamps"] = int(df["ts"].isna().sum())
    df = df.dropna(subset=["ts", "user"])

    # Exact duplicates (same learner, second, event, context, description) are logging artefacts
    n0 = len(df)
    df = df.drop_duplicates(subset=["user", "ts", "context", "component", "event", "description"])
    log["duplicates_removed"] = n0 - len(df)

    # Split "Quiz: IT_EE_Section-C" -> type / name; rows without a colon (deleted resources) -> "Other"
    parts = df["context"].fillna("").str.split(":", n=1, expand=True)
    df["context_type"] = np.where(df["context"].fillna("").str.contains(":"),
                                  parts[0].str.strip(), "Other")
    df["context_name"] = np.where(df["context"].fillna("").str.contains(":"),
                                  parts[1].str.strip(), df["context"])

    # Behavioural group + system-event flag
    df["group"] = df["content"].map(CONTENT_GROUP)
    df.loc[df["group"].isna() & df["content"].notna(), "group"] = "learning_content"
    df["group"] = df["group"].fillna("navigation")           # Content code is empty for 'System' events
    df["is_system"] = df["event"].isin(SYSTEM_EVENTS)
    log["system_events"] = int(df["is_system"].sum())

    df = df.sort_values(["user", "ts"]).reset_index(drop=True)
    log["clean_rows"] = len(df)
    return df, log


def restrict_to_course_window(df):
    """
    The export runs for ~9 months but activity collapses after the main teaching period
    (a thin tail of stragglers/re-attempts). We keep every week up to the last week in which
    >= ACTIVE_WEEK_SHARE of all learners were active; later events are reported but not modelled.
    """
    n_learners = df["user"].nunique()
    weekly_users = df.groupby(df["ts"].dt.to_period("W"))["user"].nunique() / n_learners
    last_week = weekly_users[weekly_users >= ACTIVE_WEEK_SHARE].index.max()
    course_start = df["ts"].min().normalize()
    course_end = last_week.end_time.normalize()
    main = df[df["ts"].dt.normalize() <= course_end].copy()
    main["day"] = main["ts"].dt.normalize()
    main["week"] = (main["day"] - course_start).dt.days // 7           # 0 = first course week
    info = {"course_start": course_start, "course_end": course_end,
            "n_weeks": int(main["week"].max()) + 1, "tail_events": len(df) - len(main),
            "tail_learners": df.loc[~df.index.isin(main.index), "user"].nunique()}
    return main, info

In [7]:
# --------------------------------------------------------------------------- #
# 2. SESSIONISATION
# --------------------------------------------------------------------------- #
def sessionize(main):
    """
    A session = a run of events by one learner with no gap longer than SESSION_GAP_MIN minutes.
    Auto-save pings are kept here: they prove the learner still has a quiz open.
    Duration = last event - first event (a lower bound: time on the final page is unobserved).
    """
    gap = main.groupby("user")["ts"].diff()
    new_session = gap.isna() | (gap > pd.Timedelta(minutes=SESSION_GAP_MIN))
    main["session_id"] = new_session.cumsum()
    sessions = (main.groupby("session_id")
                .agg(user=("user", "first"), start=("ts", "min"), end=("ts", "max"),
                     n_events=("ts", "size"),
                     n_interactions=("is_system", lambda s: int((~s).sum())))
                .reset_index())
    sessions["duration_min"] = (sessions["end"] - sessions["start"]).dt.total_seconds() / 60
    return main, sessions, gap


In [8]:
# --------------------------------------------------------------------------- #
# 3. FEATURE ENGINEERING
# --------------------------------------------------------------------------- #
def weekly_matrix(ia, users, n_weeks):
    """Learner x course-week matrix of deliberate interactions (zeros kept: inactivity is information)."""
    W = (ia.groupby(["user", "week"]).size().unstack(fill_value=0)
         .reindex(index=users, columns=range(n_weeks), fill_value=0))
    return W


def find_rush_window(W):
    """The two consecutive course weeks with the highest cohort-wide volume = the 'rush' window."""
    tot = W.sum(axis=0).values
    pair = tot[:-1] + tot[1:]
    start = int(np.argmax(pair))
    return (start, start + 1)


def build_features(main, sessions, info):
    """One row per learner: volume, sessions, consistency, timing, content mix, assessment actions."""
    users = pd.Index(sorted(main["user"].unique()), name="user")
    n_weeks, start, end = info["n_weeks"], info["course_start"], info["course_end"]
    ia = main[~main["is_system"]]                              # deliberate interactions only
    W = weekly_matrix(ia, users, n_weeks)
    rush = find_rush_window(W)
    F = pd.DataFrame(index=users)

    # ---- volume ----
    F["n_events"] = main.groupby("user").size()
    F["n_interactions"] = ia.groupby("user").size()

    # ---- sessions ----
    sg = sessions.groupby("user")
    F["n_sessions"] = sg.size()
    F["mean_session_min"] = sg["duration_min"].mean()
    F["median_session_min"] = sg["duration_min"].median()
    F["events_per_session"] = sg["n_events"].mean()

    # ---- consistency (weekly pattern, zeros included) ----
    days = ia.drop_duplicates(["user", "day"]).sort_values(["user", "day"])
    F["active_days"] = days.groupby("user").size()
    F["sessions_per_active_day"] = F["n_sessions"] / F["active_days"]
    gaps = days.groupby("user")["day"].diff().dt.days - 1      # inactive days between 2 active days
    F["longest_inactive_streak"] = gaps.groupby(days["user"]).max()
    Wv = W.values.astype(float)
    mean, sd = Wv.mean(1), Wv.std(1)
    with np.errstate(divide="ignore", invalid="ignore"):
        F["weekly_cv"] = np.where(mean > 0, sd / mean, 0.0)    # burstiness (higher = spikier)
        x = np.arange(n_weeks) - (n_weeks - 1) / 2
        slope = (Wv - mean[:, None]) @ x / (x ** 2).sum()
        F["trend_slope"] = np.where(mean > 0, slope / mean, 0.0)   # relative change per week
        tot = Wv.sum(1)
        F["active_week_ratio"] = (Wv > 0).mean(1)
        F["peak2_share"] = np.where(tot > 0, np.sort(Wv, 1)[:, -2:].sum(1) / tot, 0.0)
        F["rush_share"] = np.where(tot > 0, Wv[:, list(rush)].sum(1) / tot, 0.0)

    # ---- timing / earliness ----
    F["first_access_day"] = (ia.groupby("user")["day"].min() - start).dt.days
    F["days_since_last_active"] = (end - ia.groupby("user")["day"].max()).dt.days
    F["night_share"] = ia.assign(n=ia["ts"].dt.hour.isin([22, 23, 0, 1, 2, 3, 4, 5])
                                 ).groupby("user")["n"].mean()
    F["weekend_share"] = ia.assign(w=ia["ts"].dt.dayofweek >= 5).groupby("user")["w"].mean()
    assess = ia[ia["group"].isin(["graded_quiz", "assignment"])]
    F["first_assessment_day"] = (assess.groupby("user")["day"].min() - start).dt.days

    # ---- content mix (share of deliberate interactions) ----
    mix = ia.groupby(["user", "group"]).size().unstack(fill_value=0).reindex(columns=GROUPS, fill_value=0)
    mix = mix.div(mix.sum(axis=1), axis=0).add_prefix("share_")
    F = F.join(mix)

    # ---- explicit assessment actions ----
    ev = main.groupby(["user", "event"]).size().unstack(fill_value=0)
    for col, name in [("Quiz attempt started", "quiz_attempts_started"),
                      ("Quiz attempt submitted", "quiz_attempts_submitted"),
                      ("A submission has been submitted.", "assignments_submitted")]:
        F[name] = ev[col] if col in ev else 0
    F = F.reindex(users)

    # learners whose only events were system pings have no deliberate interactions
    for c in ["n_interactions", "active_days", "quiz_attempts_started", "quiz_attempts_submitted",
              "assignments_submitted", "longest_inactive_streak", "sessions_per_active_day"]:
        F[c] = F[c].fillna(0)
    F["first_access_day"] = F["first_access_day"].fillna(info["n_weeks"] * 7)
    F["days_since_last_active"] = F["days_since_last_active"].fillna(info["n_weeks"] * 7)
    F["first_assessment_day"] = F["first_assessment_day"].fillna(info["n_weeks"] * 7)
    F = F.fillna(0)
    return F, W, rush




In [9]:

# --------------------------------------------------------------------------- #
# 4. TEMPORAL VISUALISATIONS (cohort level)
# --------------------------------------------------------------------------- #
def week_label(info, w):
    return (info["course_start"] + pd.Timedelta(days=7 * int(w))).strftime("%d %b")


def savefig(fig, path):
    fig.tight_layout()
    fig.savefig(path, dpi=200)
    plt.close(fig)


def plot_cohort_timeline(ia, info, rush, fig_dir):
    """Daily deliberate interactions (+ 7-day mean) and daily active learners across the course."""
    idx = pd.date_range(info["course_start"], info["course_end"], freq="D")
    daily = ia.groupby("day").size().reindex(idx, fill_value=0)
    dau = ia.groupby("day")["user"].nunique().reindex(idx, fill_value=0)
    fig, axes = plt.subplots(2, 1, figsize=(11, 6.5), sharex=True)
    axes[0].bar(idx, daily.values, color="lightsteelblue", width=1.0, label="daily interactions")
    axes[0].plot(idx, daily.rolling(7, center=True, min_periods=1).mean(), color="navy", lw=2,
                 label="7-day mean")
    axes[1].fill_between(idx, dau.values, color="seagreen", alpha=0.35)
    axes[1].plot(idx, dau.values, color="seagreen", lw=1.2)
    r0 = info["course_start"] + pd.Timedelta(days=7 * rush[0])
    r1 = info["course_start"] + pd.Timedelta(days=7 * (rush[1] + 1))
    for ax in axes:
        ax.axvspan(r0, r1, color="tomato", alpha=0.15)
    axes[0].text(r0, axes[0].get_ylim()[1] * 0.92, " rush window", color="firebrick", va="top")
    axes[0].set(ylabel="Interactions per day", title="Cohort engagement timeline")
    axes[0].legend(loc="upper left")
    axes[1].set(ylabel="Active learners per day", xlabel="Date")
    axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
    savefig(fig, fig_dir / "cohort_timeline.png")


def plot_weekly_reach(W, info, fig_dir):
    """Weekly active learners and median interactions among the active ones."""
    active = (W > 0).sum(axis=0)
    med = W.apply(lambda c: c[c > 0].median() if (c > 0).any() else 0)
    weeks = np.arange(info["n_weeks"])
    fig, ax = plt.subplots(figsize=(10, 4.5))
    ax.bar(weeks, active, color="steelblue", alpha=0.8)
    ax.set(xlabel="Course week", ylabel="Active learners", title="Weekly reach and intensity")
    ax2 = ax.twinx()
    ax2.plot(weeks, med, color="darkorange", marker="o", ms=4)
    ax2.set_ylabel("Median interactions per active learner", color="darkorange")
    ax2.grid(False)
    ticks = weeks[::3]
    ax.set_xticks(ticks)
    ax.set_xticklabels([f"W{t}\n{week_label(info, t)}" for t in ticks], fontsize=8)
    savefig(fig, fig_dir / "weekly_reach_intensity.png")


def plot_hour_weekday(ia, fig_dir):
    """When in the week do learners work? (share of interactions per weekday x hour)"""
    h = (ia.groupby([ia["ts"].dt.dayofweek, ia["ts"].dt.hour]).size().unstack(fill_value=0)
         .reindex(index=range(7), columns=range(24), fill_value=0))
    h = h / h.values.sum()
    fig, ax = plt.subplots(figsize=(11, 4))
    sns.heatmap(h, cmap="YlOrRd", ax=ax, cbar_kws={"label": "Share of all interactions"},
                yticklabels=["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"])
    ax.set(xlabel="Hour of day", ylabel="", title="Weekly rhythm of activity")
    savefig(fig, fig_dir / "hour_weekday_heatmap.png")


def plot_activity_mix_time(ia, info, fig_dir):
    """Weekly share of interactions by behavioural group (what learners do changes through the course)."""
    m = (ia.groupby(["week", "group"]).size().unstack(fill_value=0)
         .reindex(index=range(info["n_weeks"]), columns=GROUPS, fill_value=0))
    share = m.div(m.sum(axis=1).replace(0, np.nan), axis=0).fillna(0)
    fig, ax = plt.subplots(figsize=(10, 4.8))
    ax.stackplot(share.index, share.T.values, labels=GROUPS, colors=sns.color_palette("Set2", len(GROUPS)))
    ax.set(xlim=(0, info["n_weeks"] - 1), ylim=(0, 1), xlabel="Course week",
           ylabel="Share of interactions", title="Shift in activity mix over the course")
    ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1))
    ticks = share.index[::3]
    ax.set_xticks(ticks)
    ax.set_xticklabels([f"W{t}\n{week_label(info, t)}" for t in ticks], fontsize=8)
    savefig(fig, fig_dir / "activity_mix_over_time.png")


def plot_session_diagnostics(gap, sessions, fig_dir):
    """Left: log-histogram of inter-event gaps justifies the 30-min cut. Right: session durations."""
    g = gap.dropna().dt.total_seconds() / 60
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.3))
    axes[0].hist(np.log10(g.clip(lower=1 / 60)), bins=80, color="slategray")
    axes[0].axvline(np.log10(SESSION_GAP_MIN), color="red", ls="--", label=f"{SESSION_GAP_MIN}-min threshold")
    axes[0].set(xlabel="log10(gap between consecutive events, minutes)", ylabel="Events",
                title="Inter-event gaps (bimodal: within vs. between sessions)")
    axes[0].legend()
    d = sessions.loc[sessions["duration_min"] > 0, "duration_min"]
    axes[1].hist(d.clip(upper=d.quantile(0.99)), bins=60, color="steelblue")
    axes[1].axvline(d.median(), color="red", ls="--", label=f"median = {d.median():.1f} min")
    axes[1].set(xlabel="Session duration (min, multi-event sessions, top 1% clipped)", ylabel="Sessions",
                title="Session duration")
    axes[1].legend()
    savefig(fig, fig_dir / "session_diagnostics.png")


In [10]:
# --------------------------------------------------------------------------- #
# 5. HYPOTHESIS CHECKS (descriptive, no outcome labels available)
# --------------------------------------------------------------------------- #
def test_h3_deadline_driven(F, W, rush, info, tab_dir):
    """
    H3: engagement is deadline-driven. The 2-week rush window is 2/n_weeks of the calendar;
    under evenly-spread activity it would hold ~2/n_weeks of the interactions.
    We report the observed share, the concentration ratio, and a per-learner bootstrap CI.
    """
    tot = W.values.sum()
    obs = W.values[:, list(rush)].sum() / tot
    expected = 2 / info["n_weeks"]
    rng = np.random.default_rng(SEED)
    boots = []
    Wv = W.values
    for _ in range(2000):
        s = Wv[rng.integers(0, len(Wv), len(Wv))]
        boots.append(s[:, list(rush)].sum() / s.sum())
    lo, hi = np.percentile(boots, [2.5, 97.5])
    per = F["rush_share"]
    out = pd.Series({"rush_weeks": f"W{rush[0]}-W{rush[1]}", "rush_dates_start": week_label(info, rush[0]),
                     "expected_share_if_uniform": expected, "observed_share": obs,
                     "concentration_ratio": obs / expected, "ci95_low": lo, "ci95_high": hi,
                     "median_learner_rush_share": per.median(),
                     "learners_with_rush_share_gt_2x_expected": (per > 2 * expected).mean(),
                     "learners_with_rush_share_gt_50pct": (per > 0.5).mean(),
                     "median_peak2_share": F["peak2_share"].median()})
    out.to_csv(tab_dir / "h3_deadline_concentration.csv", header=["value"])
    print("\n[H3] deadline concentration\n",
          out.map(lambda v: round(v, 3) if isinstance(v, (float, np.floating)) else v).to_string())
    return out



In [11]:
# --------------------------------------------------------------------------- #
# 6. MACHINE LEARNING: LEARNER SEGMENTATION
# --------------------------------------------------------------------------- #
def prepare_matrix(F):
    """log1p the right-skewed features, then z-score every feature."""
    X = F[CLUSTER_FEATURES].copy()
    for c in LOG_FEATURES:
        X[c] = np.log1p(X[c])
    scaler = StandardScaler()
    return scaler.fit_transform(X), X.columns.tolist()


def score_k(Z):
    rows = []
    for k in K_RANGE:
        km = KMeans(k, n_init=30, random_state=SEED).fit(Z)
        rows.append({"k": k, "inertia": km.inertia_, "silhouette": silhouette_score(Z, km.labels_),
                     "calinski_harabasz": calinski_harabasz_score(Z, km.labels_),
                     "davies_bouldin": davies_bouldin_score(Z, km.labels_),
                     "smallest_cluster": int(np.bincount(km.labels_).min())})
    return pd.DataFrame(rows)


def plot_k_selection(res, k_star, fig_dir):
    fig, axes = plt.subplots(1, 4, figsize=(16, 3.6))
    for ax, col, title in zip(axes, ["inertia", "silhouette", "calinski_harabasz", "davies_bouldin"],
                              ["Inertia (elbow)", "Silhouette (higher better)",
                               "Calinski-Harabasz (higher better)", "Davies-Bouldin (lower better)"]):
        ax.plot(res["k"], res[col], "o-")
        ax.axvline(k_star, color="red", ls="--", lw=1)
        ax.set(title=title, xlabel="k")
    savefig(fig, fig_dir / "k_selection.png")


def bootstrap_stability(Z, k, ref_labels):
    """Re-cluster 80% subsamples; ARI between the subsample clustering and the full-data labels."""
    rng = np.random.default_rng(SEED)
    aris = []
    for _ in range(N_BOOT):
        idx = rng.choice(len(Z), int(0.8 * len(Z)), replace=False)
        lab = KMeans(k, n_init=10, random_state=int(rng.integers(1e6))).fit_predict(Z[idx])
        aris.append(adjusted_rand_score(ref_labels[idx], lab))
    return np.array(aris)


def permutation_null_silhouette(Z, k):
    """Shuffle each feature independently (keeps marginals, destroys joint structure)."""
    rng = np.random.default_rng(SEED)
    null = []
    for _ in range(N_PERM):
        Zp = np.column_stack([rng.permutation(Z[:, j]) for j in range(Z.shape[1])])
        null.append(silhouette_score(Zp, KMeans(k, n_init=5, random_state=SEED).fit_predict(Zp)))
    return np.array(null)


def eta_squared(Fc, labels, cols):
    """Share of each feature's variance explained by cluster membership (descriptive effect size)."""
    rows = []
    for c in cols:
        x = Fc[c].values.astype(float)
        ss_tot = ((x - x.mean()) ** 2).sum()
        ss_b = sum(len(x[labels == g]) * (x[labels == g].mean() - x.mean()) ** 2 for g in np.unique(labels))
        rows.append({"feature": c, "family": FEATURE_FAMILY[c], "eta_squared": ss_b / ss_tot if ss_tot else 0})
    return pd.DataFrame(rows).sort_values("eta_squared", ascending=False)


def run_clustering(F, W, info, rush, fig_dir, tab_dir):
    eligible = F["n_interactions"] >= MIN_INTERACTIONS
    Fc = F[eligible].copy()
    print(f"\n[ML] clustering {len(Fc)} learners with >= {MIN_INTERACTIONS} interactions "
          f"({(~eligible).sum()} below the threshold are excluded)")
    Z, cols = prepare_matrix(Fc)

    # ---- choose k ----
    res = score_k(Z)
    k_star = int(res[res["k"].isin(K_CANDIDATES)].sort_values("silhouette", ascending=False).iloc[0]["k"])
    res.round(4).to_csv(tab_dir / "k_selection.csv", index=False)
    print(res.round(3).to_string(index=False), f"\n  -> chosen k = {k_star}")
    plot_k_selection(res, k_star, fig_dir)

    # ---- final K-Means; relabel so cluster 1 = highest median volume ----
    km = KMeans(k_star, n_init=100, random_state=SEED).fit(Z)
    order = pd.Series(Fc["n_interactions"].values).groupby(km.labels_).median().sort_values(ascending=False).index
    remap = {old: new + 1 for new, old in enumerate(order)}
    labels = np.array([remap[l] for l in km.labels_])
    Fc["cluster"] = labels

    # ---- validation ----
    hier = AgglomerativeClustering(n_clusters=k_star, linkage="ward").fit_predict(Z)
    ari_h = adjusted_rand_score(labels, hier)
    boot = bootstrap_stability(Z, k_star, labels)
    sil = silhouette_score(Z, labels)
    null = permutation_null_silhouette(Z, k_star)
    p_perm = (1 + (null >= sil).sum()) / (1 + len(null))
    val = pd.Series({"k": k_star, "silhouette": sil, "calinski_harabasz": calinski_harabasz_score(Z, labels),
                     "davies_bouldin": davies_bouldin_score(Z, labels),
                     "ARI_kmeans_vs_ward": ari_h, "bootstrap_ARI_mean": boot.mean(),
                     "bootstrap_ARI_p05": np.percentile(boot, 5), "null_silhouette_mean": null.mean(),
                     "null_silhouette_p95": np.percentile(null, 95), "permutation_p": p_perm})
    val.to_csv(tab_dir / "clustering_validation.csv", header=["value"])
    print("\n[H4] validation\n", val.round(3).to_string())
    Fc["cluster_ward"] = hier

    # ---- profiles ----
    prof_cols = CLUSTER_FEATURES + ["n_sessions", "active_days", "median_session_min", "events_per_session",
                                    "trend_slope", "peak2_share", "days_since_last_active",
                                    "first_assessment_day", "quiz_attempts_submitted", "assignments_submitted",
                                    "share_assignment", "share_social", "share_navigation"]
    prof = Fc.groupby("cluster")[prof_cols].median()
    prof.insert(0, "n_learners", Fc.groupby("cluster").size())
    prof.insert(1, "share_of_learners", prof["n_learners"] / len(Fc))
    prof.round(3).to_csv(tab_dir / "cluster_profiles_median.csv")
    Fc.groupby("cluster")[prof_cols].mean().round(3).to_csv(tab_dir / "cluster_profiles_mean.csv")
    eta = eta_squared(Fc, labels, CLUSTER_FEATURES)
    eta.round(4).to_csv(tab_dir / "feature_eta_squared.csv", index=False)
    fam = eta.groupby("family")["eta_squared"].mean().sort_values(ascending=False)
    print("\n[H4] mean eta^2 by feature family\n", fam.round(3).to_string())
    print("\n[ML] median profile per cluster\n", prof[["n_learners"] + CLUSTER_FEATURES].round(2).T.to_string())

    # ---- figures ----
    plot_cluster_figures(Fc, Z, cols, W, info, rush, km, labels, fig_dir)
    # ---- hierarchical dendrogram ----
    fig, ax = plt.subplots(figsize=(11, 4.5))
    dendrogram(linkage(Z, "ward"), truncate_mode="lastp", p=30, ax=ax, color_threshold=None,
               above_threshold_color="grey")
    ax.set(title="Ward hierarchical clustering (last 30 merges)", xlabel="Merged groups (size)",
           ylabel="Ward distance")
    savefig(fig, fig_dir / "dendrogram.png")

    out = F.join(Fc[["cluster"]])
    out.to_csv(tab_dir / "learner_features_with_clusters.csv")
    return Fc, val, prof, eta


def cluster_palette(k):
    return dict(zip(range(1, k + 1), sns.color_palette("colorblind", k)))


def plot_cluster_figures(Fc, Z, cols, W, info, rush, km, labels, fig_dir):
    k = len(np.unique(labels))
    pal = cluster_palette(k)
    sizes = Fc.groupby("cluster").size()
    names = {c: f"C{c} (n={sizes[c]})" for c in sizes.index}

    # 1. PCA scatter
    pca = PCA(2, random_state=SEED).fit(Z)
    P = pca.transform(Z)
    fig, ax = plt.subplots(figsize=(7, 5.5))
    for c in sorted(pal):
        m = labels == c
        ax.scatter(P[m, 0], P[m, 1], s=35, color=pal[c], label=names[c], alpha=0.85, edgecolor="white", lw=0.4)
    ax.set(xlabel=f"PC1 ({pca.explained_variance_ratio_[0]:.0%} var.)",
           ylabel=f"PC2 ({pca.explained_variance_ratio_[1]:.0%} var.)",
           title="Learner segments in PCA space")
    ax.legend()
    savefig(fig, fig_dir / "cluster_pca.png")

    # 2. profile heatmap: cluster-mean z-score per feature
    Zdf = pd.DataFrame(Z, columns=cols)
    Zdf["cluster"] = labels
    heat = Zdf.groupby("cluster")[cols].mean()
    heat.index = [names[c] for c in heat.index]
    fig, ax = plt.subplots(figsize=(11, 0.9 * k + 2.2))
    sns.heatmap(heat, cmap="RdBu_r", center=0, annot=True, fmt=".1f", ax=ax,
                cbar_kws={"label": "Mean z-score"})
    ax.set(title="Behavioural profile of each segment (standardised features)")
    plt.setp(ax.get_xticklabels(), rotation=35, ha="right")
    savefig(fig, fig_dir / "cluster_profile_heatmap.png")

    # 3. temporal: weekly trajectory per cluster (mean interactions per learner)
    Wc = W.loc[Fc.index]
    fig, ax = plt.subplots(figsize=(10, 5))
    wk = np.arange(info["n_weeks"])
    for c in sorted(pal):
        ax.plot(wk, Wc[Fc["cluster"] == c].mean(), color=pal[c], lw=2, label=names[c])
    ax.axvspan(rush[0], rush[1], color="tomato", alpha=0.12)
    ax.set_yscale("symlog", linthresh=10)
    ax.set(xlabel="Course week", ylabel="Mean interactions per learner (symlog)",
           title="Weekly engagement trajectory of each segment")
    ax.set_xticks(wk[::3])
    ax.set_xticklabels([f"W{t}\n{week_label(info, t)}" for t in wk[::3]], fontsize=8)
    ax.legend()
    savefig(fig, fig_dir / "cluster_weekly_trajectories.png")

    # 4. temporal: cumulative share of each segment's interactions (reveals crammers vs steady)
    fig, ax = plt.subplots(figsize=(8, 5))
    for c in sorted(pal):
        cum = Wc[Fc["cluster"] == c].sum(axis=0).cumsum()
        ax.plot(wk, cum / cum.iloc[-1], color=pal[c], lw=2, label=names[c])
    ax.plot([0, wk[-1]], [0, 1], "k--", lw=1, label="perfectly even")
    ax.axvspan(rush[0], rush[1], color="tomato", alpha=0.12)
    ax.set(xlabel="Course week", ylabel="Cumulative share of segment's interactions",
           title="When does each segment do its work?")
    ax.legend()
    savefig(fig, fig_dir / "cluster_cumulative_activity.png")

    # 5. learner x week "barcode" sorted by cluster
    order = Fc.sort_values(["cluster", "n_interactions"], ascending=[True, False]).index
    M = np.log1p(W.loc[order].values)
    fig, (axc, ax) = plt.subplots(1, 2, figsize=(12, 7), gridspec_kw={"width_ratios": [0.03, 1], "wspace": 0.02})
    cl = Fc.loc[order, "cluster"].values
    axc.imshow(cl[:, None], aspect="auto", cmap=matplotlib.colors.ListedColormap([pal[c] for c in sorted(pal)]))
    axc.axis("off")
    im = ax.imshow(M, aspect="auto", cmap="magma_r", interpolation="nearest")
    ax.set(xlabel="Course week", yticks=[], title="Every learner's weekly activity, grouped by segment (log scale)")
    ax.set_xticks(wk[::3])
    ax.set_xticklabels([f"W{t}" for t in wk[::3]])
    b = np.where(np.diff(cl) != 0)[0] + 0.5
    for y in b:
        ax.axhline(y, color="white", lw=2)
    fig.colorbar(im, ax=ax, label="log(1 + interactions)", fraction=0.03, pad=0.01)
    savefig(fig, fig_dir / "cluster_learner_week_heatmap.png")

    # 6. hour-of-day rhythm per cluster
    return None


def plot_cluster_hourly(ia, Fc, fig_dir):
    k = Fc["cluster"].nunique()
    pal = cluster_palette(k)
    d = ia[ia["user"].isin(Fc.index)].copy()
    d["cluster"] = d["user"].map(Fc["cluster"])
    h = d.groupby(["cluster", d["ts"].dt.hour]).size().unstack(fill_value=0).reindex(columns=range(24), fill_value=0)
    h = h.div(h.sum(axis=1), axis=0)
    fig, ax = plt.subplots(figsize=(9, 4.5))
    for c in h.index:
        ax.plot(h.columns, h.loc[c], color=pal[c], lw=2, label=f"C{c}")
    ax.set(xlabel="Hour of day", ylabel="Share of segment's interactions", title="Daily rhythm by segment")
    ax.legend()
    savefig(fig, fig_dir / "cluster_hourly_rhythm.png")




In [12]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

def find_csv(cli_path):
    # Adjust this path if the file is inside a specific subfolder in your Drive
    colab_path = "/content/drive/MyDrive/MOODLE Data.csv"
    return colab_path

# --------------------------------------------------------------------------- #
# 7. MAIN
# --------------------------------------------------------------------------- #
def main():
    import argparse
    ap = argparse.ArgumentParser()
    ap.add_argument("--csv", default=None, help="path to the Moodle export (default: auto-detect)")
    ap.add_argument("--out", default="outputs/moodle", help="output folder")

    # Use parse_known_args to prevent notebook/IPython kernel arguments from crashing the script
    args, _ = ap.parse_known_args()

    out = Path(args.out)
    fig_dir, tab_dir = out / "figures", out / "tables"
    for d in (fig_dir, tab_dir):
        d.mkdir(parents=True, exist_ok=True)

    print("[1/6] Loading and cleaning ...")
    df, log = load_and_clean(find_csv(args.csv))
    main_df, info = restrict_to_course_window(df)
    print("  cleaning log:", log)
    print(f"  course window: {info['course_start'].date()} -> {info['course_end'].date()} "
          f"({info['n_weeks']} weeks); {info['tail_events']:,} later events excluded")

    print("[2/6] Sessionising ...")
    main_df, sessions, gap = sessionize(main_df)
    ia = main_df[~main_df["is_system"]]
    print(f"  {main_df['user'].nunique()} learners, {len(main_df):,} events "
          f"({len(ia):,} deliberate interactions), {len(sessions):,} sessions")

    print("[3/6] Engineering features ...")
    F, W, rush = build_features(main_df, sessions, info)
    F.round(4).to_csv(tab_dir / "learner_features.csv")
    summary = pd.Series({**log, "course_start": info["course_start"].date(), "course_end": info["course_end"].date(),
                         "n_weeks": info["n_weeks"], "events_after_course_window": info["tail_events"],
                         "learners_in_window": main_df["user"].nunique(), "events_in_window": len(main_df),
                         "interactions_in_window": len(ia), "sessions": len(sessions),
                         "median_session_min_multi_event": sessions.loc[sessions.duration_min > 0, "duration_min"].median(),
                         "single_event_sessions_share": (sessions.duration_min == 0).mean(),
                         "median_sessions_per_learner": F["n_sessions"].median(),
                         "median_interactions_per_learner": F["n_interactions"].median()})
    summary.to_csv(tab_dir / "dataset_summary.csv", header=["value"])
    F.describe().T.round(3).to_csv(tab_dir / "feature_descriptives.csv")
    print(F[CLUSTER_FEATURES].describe().T.round(2).to_string())

    print("[4/6] Temporal visualisations ...")
    plot_cohort_timeline(ia, info, rush, fig_dir)
    plot_weekly_reach(W, info, fig_dir)
    plot_hour_weekday(ia, fig_dir)
    plot_activity_mix_time(ia, info, fig_dir)
    plot_session_diagnostics(gap, sessions, fig_dir)
    test_h3_deadline_driven(F, W, rush, info, tab_dir)
    ev = ia.groupby("event").size().sort_values(ascending=False).head(15)
    ev.to_csv(tab_dir / "top_events.csv", header=["count"])

    print("[5/6] Clustering ...")
    Fc, val, prof, eta = run_clustering(F, W, info, rush, fig_dir, tab_dir)
    plot_cluster_hourly(ia, Fc, fig_dir)

    print("[6/6] Done.")
    print(f"\nFigures -> {fig_dir}\nTables  -> {tab_dir}")

if __name__ == "__main__":
    main()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[1/6] Loading and cleaning ...
  cleaning log: {'raw_rows': 146048, 'raw_learners': 174, 'unparseable_timestamps': 0, 'duplicates_removed': 0, 'system_events': 40846, 'clean_rows': 146048}
  course window: 2025-10-23 -> 2026-04-12 (25 weeks); 497 later events excluded
[2/6] Sessionising ...
  173 learners, 145,551 events (104,705 deliberate interactions), 6,595 sessions
[3/6] Engineering features ...
                         count    mean     std  min     25%     50%     75%      max
n_interactions           173.0  605.23  507.37  0.0  294.00  509.00  831.00  3699.00
mean_session_min         173.0   12.74    8.04  0.0    7.80   10.70   15.47    39.76
active_week_ratio        173.0    0.50    0.24  0.0    0.36    0.52    0.68     1.00
weekly_cv                173.0    2.46    0.99  0.0    1.75    2.23    2.96     4.90
longest_inactive_streak  173.0   27.73   1

/tmp/ipykernel_1299/1912374454.py:9: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()


[6/6] Done.

Figures -> outputs/moodle/figures
Tables  -> outputs/moodle/tables
